# Assignment 6: Vision and Language

**EECS 183/283A · Due Fri Dec 4, 11:59 PM PT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zinengtang/cs183-fall26-a6/blob/main/HW6.ipynb)

You will implement CLIP's contrastive loss and zero-shot classifier (Part 1), test CLIP on grounding challenges and build a modular system that fixes them (Part 2), and implement the Rational Speech Acts model of pragmatics (Part 3). Optional Part 4 tests a small modern VLM on the same challenges. There is no training and no GPU: a free Colab CPU runtime is enough.

| Part | Topic | Points |
|---|---|---|
| 1 | CLIP | 35 |
| 2 | Where CLIP breaks, and a modular fix | 40 |
| 3 | Pragmatics (RSA) | 25 |
| 4 | Does a modern VLM fix Part 2? (optional) | +10 extra credit |

**Rules**
- Write code only at `# YOUR CODE HERE`, and wherever a question asks you to edit a cell.
- `# GRADED:` cells are autograded, each one on its own. Keep the header line and the function names, put imports inside the cell, and don't use variables from other cells.
- Answer the ★ questions in the **Your answer** cells. They are graded from your PDF.
- **Submit** `HW6.ipynb` and `HW6.pdf` (*File → Print → Save as PDF*) to Gradescope **A6**.

Collaborators: ______

## Setup

In [ ]:
# Colab has everything except possibly `datasets`. Locally: pip install -r requirements.txt
import importlib.util, subprocess, sys
if importlib.util.find_spec("datasets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print("device:", device, "| torch", torch.__version__)

---
# Part 1: CLIP (35 pts)

For a batch of $N$ (image, caption) pairs with unit-normalized embeddings $\mathbf{x}_i$ and $\mathbf{y}_j$, CLIP's logits are $\ell_{ij} = s\,\mathbf{x}_i^\top \mathbf{y}_j$, where $s$ is a learned **logit scale**. The loss averages cross-entropy in both directions: each image picks its caption, and each caption picks its image.

$$\mathcal{L} = \frac{1}{2}\left[\frac{1}{N}\sum_{i} -\log \frac{e^{\ell_{ii}}}{\sum_{j} e^{\ell_{ij}}} \;+\; \frac{1}{N}\sum_{j} -\log \frac{e^{\ell_{jj}}}{\sum_{i} e^{\ell_{ij}}}\right]$$

## 1.1 Contrastive loss (8 pts, autograded)

Implement `clip_contrastive_loss` using `F.cross_entropy`. Normalize the inputs yourself. `logit_scale` is $s$ itself (for example `100.0`), not $\log s$.

In [ ]:
# GRADED: clip_contrastive_loss
import torch
import torch.nn.functional as F


def clip_contrastive_loss(image_emb, text_emb, logit_scale):
    """Symmetric CLIP / InfoNCE loss.

    Args:
        image_emb:   float tensor [N, D], row i is the i-th image embedding (not necessarily unit norm)
        text_emb:    float tensor [N, D], row i is the caption paired with image i
        logit_scale: python float or 0-dim tensor s (the multiplier, NOT log s)
    Returns:
        0-dim tensor: the average of the image->text and text->image cross-entropy losses.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Sanity check (not graded): random pairs at s=1 give about log N; aligned pairs at s=100 give about 0.
torch.manual_seed(0)
N, D = 16, 32
a = torch.randn(N, D)
print("random, s=1   :", clip_contrastive_loss(a, torch.randn(N, D), 1.0).item(), " log N =", math.log(N))
print("aligned, s=100:", clip_contrastive_loss(a, 3 * a, 100.0).item())

## Load CLIP and CIFAR-10

We use OpenAI's ViT-B/32 CLIP (about 600 MB), the only model in Parts 1–3, and 1,000 CIFAR-10 test images.

In [ ]:
from transformers import CLIPModel, CLIPProcessor


def load_clip(name):
    model = CLIPModel.from_pretrained(name).to(device).eval()
    processor = CLIPProcessor.from_pretrained(name)
    return model, processor


def _as_tensor(out):
    return out if torch.is_tensor(out) else out.pooler_output


@torch.no_grad()
def encode_images(model, processor, images, batch_size=100):
    feats = []
    for i in range(0, len(images), batch_size):
        x = processor(images=images[i:i + batch_size], return_tensors="pt").to(device)
        feats.append(_as_tensor(model.get_image_features(**x)).float().cpu())
    return torch.cat(feats)


@torch.no_grad()
def encode_texts(model, processor, texts, batch_size=256):
    feats = []
    for i in range(0, len(texts), batch_size):
        x = processor(text=list(texts[i:i + batch_size]), return_tensors="pt", padding=True).to(device)
        feats.append(_as_tensor(model.get_text_features(**x)).float().cpu())
    return torch.cat(feats)


clip_b32, proc_b32 = load_clip("openai/clip-vit-base-patch32")
LOGIT_SCALE = clip_b32.logit_scale.exp().item()
print("learned logit scale s =", round(LOGIT_SCALE, 2))

In [ ]:
from datasets import load_dataset

cifar = load_dataset("uoft-cs/cifar10", split="test",                 # test split only (24 MB)
                     data_files={"test": "plain_text/test-00000-of-00001.parquet"})
CLASSES = cifar.features["label"].names
subset = np.random.RandomState(0).choice(len(cifar), 1000, replace=False)
cifar_images = [cifar[int(i)]["img"].convert("RGB") for i in subset]
cifar_labels = np.array([cifar[int(i)]["label"] for i in subset])
print(CLASSES)

t0 = time.time()
cifar_img_emb = encode_images(clip_b32, proc_b32, cifar_images)      # [1000, 512], NOT normalized
print("encoded", tuple(cifar_img_emb.shape), f"in {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img, y in zip(axes, cifar_images, cifar_labels):
    ax.imshow(img); ax.set_title(CLASSES[y], fontsize=9); ax.axis("off")
plt.show()

### ★ Q1.1 In-batch negatives (4 pts)

The cell below computes your loss on two batches. Batch A has one image per class. Batch B has 5 airplanes and 5 ships. Every caption is `"a photo of a {class}."`.

Report both losses. Why is batch B's loss much higher, even though CLIP classifies these images well? What does the loss assume about the other items in a batch, and when does that assumption fail for web (image, alt-text) data?

In [ ]:
caption = lambda y: f"a photo of a {CLASSES[y]}."
batch_a = [int(np.where(cifar_labels == k)[0][0]) for k in range(10)]
batch_b = list(np.where(cifar_labels == 0)[0][:5]) + list(np.where(cifar_labels == 8)[0][:5])
for name, b in [("A (10 distinct classes)", batch_a), ("B (5 airplanes + 5 ships)", batch_b)]:
    txt = encode_texts(clip_b32, proc_b32, [caption(cifar_labels[i]) for i in b])
    print(f"batch {name}: loss = {clip_contrastive_loss(cifar_img_emb[b], txt, LOGIT_SCALE).item():.3f}")

**Your answer (Q1.1):**

*Write your answer here.*

## 1.2 Zero-shot classification (9 pts, autograded)

Implement two functions:

- `build_zero_shot_weights(class_names, templates, encode_text_fn)` → `[K, D]`. For each class, encode all of its filled-in templates in one call, **normalize each embedding**, average them, and re-normalize.
- `zero_shot_logits(image_emb, class_weights, logit_scale)` → `[N, K]` logits, equal to $s \cdot \cos(\text{image}, \text{class})$.

In [ ]:
# GRADED: zero_shot
import torch
import torch.nn.functional as F


def build_zero_shot_weights(class_names, templates, encode_text_fn):
    """
    Args:
        class_names:    list of K strings, e.g. ["airplane", "automobile", ...]
        templates:      list of T format strings containing "{}", e.g. ["a photo of a {}."]
        encode_text_fn: function list[str] -> float tensor [n, D] (unnormalized)
    Returns:
        float tensor [K, D]; row k is the unit-norm (prompt-ensembled) embedding of class k.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def zero_shot_logits(image_emb, class_weights, logit_scale):
    """
    Args:
        image_emb:     float tensor [N, D] (unnormalized)
        class_weights: float tensor [K, D] (unit norm rows, from build_zero_shot_weights)
        logit_scale:   float s
    Returns:
        float tensor [N, K] of logits s * cos(image_n, class_k).
    """
    # YOUR CODE HERE
    raise NotImplementedError

We compare three prompt settings: the bare class name, one template, and 8 templates from the CLIP paper.

In [ ]:
TEMPLATE_SETS = {
    "bare name": ["{}"],
    "a photo of a {}.": ["a photo of a {}."],
    "8-template ensemble": [
        "a photo of a {}.", "a blurry photo of a {}.", "a low resolution photo of a {}.",
        "a photo of the small {}.", "a photo of the big {}.", "a pixelated photo of a {}.",
        "a cropped photo of a {}.", "a good photo of a {}.",
    ],
}
encode_text_b32 = lambda texts: encode_texts(clip_b32, proc_b32, texts)

zs_results = {}
for name, templates in TEMPLATE_SETS.items():
    W = build_zero_shot_weights(CLASSES, templates, encode_text_b32)
    logits = zero_shot_logits(cifar_img_emb, W, LOGIT_SCALE)
    zs_results[name] = (logits, (logits.argmax(1).numpy() == cifar_labels).mean())
    print(f"{name:>22s}: zero-shot accuracy = {zs_results[name][1]:.3f}")

### ★ Q1.2 Logit scale (4 pts)

The cell below rescales the ensemble logits to different values of $s$. It reports accuracy, mean confidence, and calibration error (ECE).

(a) Why does accuracy stay fixed while confidence changes?

(b) Which $s$ is best calibrated? Why might it differ from the $s$ learned in training? Recall that cosines lie in $[-1, 1]$.

(c) CLIP caps $s$ at 100. What would happen to the training loss and its gradients if $s$ were fixed at 1?

In [ ]:
def ece(probs, labels, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    bins = np.linspace(0, 1, n_bins + 1)
    err = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            err += m.mean() * abs((pred[m] == labels[m]).mean() - conf[m].mean())
    return err

cos_ens = zs_results["8-template ensemble"][0] / LOGIT_SCALE
print(f"{'s':>6s} {'acc':>6s} {'conf':>6s} {'ECE':>6s}")
for s in [1, 10, 30, 50, LOGIT_SCALE, 300]:
    p = torch.softmax(s * cos_ens, dim=1).numpy()
    print(f"{s:6.0f} {(p.argmax(1) == cifar_labels).mean():6.3f} {p.max(1).mean():6.3f} {ece(p, cifar_labels):6.3f}")

**Your answer (Q1.2):**

*Write your answer here.*

### ★ Q1.3 Prompts (5 pts)

(a) Report the three accuracies above, and give one reason ensembling helps.

(b) Edit `my_templates`, report the accuracy you get, and explain your choice in 2–3 sentences. A thoughtful attempt gets full credit, even if it is lower.

In [ ]:
my_templates = ["a photo of a {}.", "a close-up photo of a {}."]   # <- edit me
W_mine = build_zero_shot_weights(CLASSES, my_templates, encode_text_b32)
acc_mine = (zero_shot_logits(cifar_img_emb, W_mine, LOGIT_SCALE).argmax(1).numpy() == cifar_labels).mean()
print(f"my templates: zero-shot accuracy = {acc_mine:.3f}")

**Your answer (Q1.3):**

*Write your answer here.*

### ★ Q1.4 Error analysis (5 pts)

(a) Name the two most-confused class pairs.

(b) Explain each pair. Consider the image content, the 32×32 resolution, and what web captions say about these classes.

(c) Name one limitation of zero-shot classification compared with a linear probe trained on CIFAR-10.

In [ ]:
pred = zs_results["8-template ensemble"][0].argmax(1).numpy()
cm = np.zeros((10, 10), dtype=int)
for y, p in zip(cifar_labels, pred):
    cm[y, p] += 1

fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

off = cm + cm.T
np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(i + 1, 10)], reverse=True)[:4]
print("most-confused pairs (both directions):", pairs)

wrong = np.where(pred != cifar_labels)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(12, 1.9))
for ax, i in zip(axes, wrong):
    ax.imshow(cifar_images[i]); ax.axis("off")
    ax.set_title(f"{CLASSES[cifar_labels[i]]}\n→ {CLASSES[pred[i]]}", fontsize=8)
plt.show()

**Your answer (Q1.4):**

*Write your answer here.*

---
# Part 2: Where CLIP breaks, and a modular fix (40 pts)

The lecture listed grounding challenges for vision–language models. Here we test CLIP on four of them with rendered scenes. Each probe is an image with a **true** caption and a minimally different **foil**. CLIP passes a probe if it scores the true caption higher than the foil.

| challenge | true caption (example) | foil |
|---|---|---|
| binding | a red circle and a blue square | a blue circle and a red square |
| spatial | a red circle above a blue square | a red circle below a blue square |
| counting | three circles | two circles |
| negation | no red shapes | a red shape |

Every caption also comes with a small **program** that states what it means. Section 2.2 uses these programs.

In [ ]:
COLORS = {"red": (220, 30, 30), "blue": (30, 70, 220), "green": (30, 160, 60), "yellow": (235, 200, 20)}
SHAPES = ["circle", "square", "triangle"]
NUM_WORDS = {1: "one", 2: "two", 3: "three", 4: "four"}
KINDS = ["binding", "spatial", "counting", "negation"]
CELL = 224 // 3                                             # scenes are laid out on a 3 x 3 grid


def draw_shape(draw, color, shape, cy, cx, r):
    fill = COLORS[color]
    if shape == "circle":
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fill)
    elif shape == "square":
        draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=fill)
    else:
        draw.polygon([(cx, cy - r), (cx - r, cy + r * 0.8), (cx + r, cy + r * 0.8)], fill=fill)


def render_scene(objects, rng):
    """objects: list of (color, shape, row, col). Returns the image and the true object list."""
    img = Image.new("RGB", (224, 224), "white")
    draw = ImageDraw.Draw(img)
    truth = []
    for color, shape, row, col in objects:
        r = rng.randint(17, 23)
        cy = CELL * row + CELL // 2 + rng.randint(-6, 7)
        cx = CELL * col + CELL // 2 + rng.randint(-6, 7)
        draw_shape(draw, color, shape, cy, cx, r)
        truth.append({"color": color, "shape": shape, "box": (cy - r, cx - r, cy + r, cx + r)})
    return img, truth


def make_probe(kind, i, rng):
    """One probe: an image, a true caption and a foil, each with its program."""
    pick = lambda xs, k: [str(x) for x in rng.choice(xs, k, replace=False)]
    cells = [(int(c) // 3, int(c) % 3) for c in rng.choice(9, 9, replace=False)]
    if kind == "binding":
        (c1, c2), (s1, s2) = pick(list(COLORS), 2), pick(SHAPES, 2)
        img, objs = render_scene([(c1, s1, *cells[0]), (c2, s2, *cells[1])], rng)
        true, foil = f"a {c1} {s1} and a {c2} {s2}", f"a {c2} {s1} and a {c1} {s2}"
        exists = lambda c, s: {"op": "exists", "color": c, "shape": s}
        prog_true = {"op": "and", "args": [exists(c1, s1), exists(c2, s2)]}
        prog_foil = {"op": "and", "args": [exists(c2, s1), exists(c1, s2)]}
    elif kind == "spatial":
        (c1, c2), (s1, s2) = pick(list(COLORS), 2), pick(SHAPES, 2)
        col, (r1, r2) = int(rng.randint(3)), sorted(int(r) for r in rng.choice(3, 2, replace=False))
        img, objs = render_scene([(c1, s1, r1, col), (c2, s2, r2, col)], rng)   # (c1, s1) is above
        rel, other = ("above", "below") if i % 2 == 0 else ("below", "above")
        a, b = ({"color": c1, "shape": s1}, {"color": c2, "shape": s2}) if rel == "above" else \
               ({"color": c2, "shape": s2}, {"color": c1, "shape": s1})
        true = f"a {a['color']} {a['shape']} {rel} a {b['color']} {b['shape']}"
        foil = f"a {a['color']} {a['shape']} {other} a {b['color']} {b['shape']}"
        prog_true = {"op": "relation", "rel": rel, "a": a, "b": b}
        prog_foil = {"op": "relation", "rel": other, "a": a, "b": b}
    elif kind == "counting":
        s = pick(SHAPES, 1)[0]
        n, n_other = int(rng.randint(1, 5)), int(rng.randint(0, 2))
        others = [x for x in SHAPES if x != s]
        spec = [(pick(list(COLORS), 1)[0], s) for _ in range(n)] + \
               [(pick(list(COLORS), 1)[0], pick(others, 1)[0]) for _ in range(n_other)]
        img, objs = render_scene([(c, sh, *cells[j]) for j, (c, sh) in enumerate(spec)], rng)
        n_foil = n + 1 if (n == 1 or (n < 4 and i % 2 == 0)) else n - 1
        words = lambda k: f"{NUM_WORDS[k]} {s}" + ("" if k == 1 else "s")
        true, foil = words(n), words(n_foil)
        prog_true, prog_foil = {"op": "count", "shape": s, "n": n}, {"op": "count", "shape": s, "n": n_foil}
    else:  # negation: half of the images contain the color, half do not
        c = pick(list(COLORS), 1)[0]
        rest = [x for x in COLORS if x != c]
        k = int(rng.randint(2, 4))
        spec = [(pick(rest, 1)[0], pick(SHAPES, 1)[0]) for _ in range(k)]
        if i % 2 == 0:
            spec[0] = (c, spec[0][1])
        img, objs = render_scene([(cc, sh, *cells[j]) for j, (cc, sh) in enumerate(spec)], rng)
        some, none = (f"a {c} shape", {"op": "exists", "color": c}), (f"no {c} shapes", {"op": "not", "arg": {"op": "exists", "color": c}})
        (true, prog_true), (foil, prog_foil) = (some, none) if i % 2 == 0 else (none, some)
    return {"kind": kind, "image": img, "objects": objs, "true": true, "foil": foil,
            "prog_true": prog_true, "prog_foil": prog_foil}


probe_rng = np.random.RandomState(0)
PROBES = [make_probe(kind, i, probe_rng) for kind in KINDS for i in range(100)]
probe_kind = np.array([p["kind"] for p in PROBES])

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, kind in zip(axes, KINDS):
    p = PROBES[KINDS.index(kind) * 100]
    ax.imshow(p["image"]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{kind}\n✓ {p['true']}\n✗ {p['foil']}", fontsize=8)
plt.tight_layout(); plt.show()

## 2.1 Testing CLIP

In [ ]:
probe_img = F.normalize(encode_images(clip_b32, proc_b32, [p["image"] for p in PROBES]), dim=-1)
probe_true = F.normalize(encode_texts(clip_b32, proc_b32, [p["true"] for p in PROBES]), dim=-1)
probe_foil = F.normalize(encode_texts(clip_b32, proc_b32, [p["foil"] for p in PROBES]), dim=-1)
clip_ok = ((probe_img * probe_true).sum(1) > (probe_img * probe_foil).sum(1)).numpy()
text_cos = (probe_true * probe_foil).sum(1).numpy()

display(pd.DataFrame({"CLIP accuracy": [clip_ok[probe_kind == k].mean() for k in KINDS],
                      "text-only cos(true, foil)": [text_cos[probe_kind == k].mean() for k in KINDS]},
                     index=KINDS).round(3))

# How often does CLIP prefer one wording, whatever the image shows?
def preference(kind, word):
    idx = np.where(probe_kind == kind)[0]
    true_has = np.array([word in PROBES[i]["true"] for i in idx])
    picked = np.array([clip_ok[i] == true_has[j] for j, i in enumerate(idx)])   # picked the caption with `word`
    return picked[true_has].mean(), picked[~true_has].mean()

for kind, word in [("spatial", " above "), ("negation", "no ")]:
    when_true, when_false = preference(kind, word)
    print(f"{kind}: CLIP picks the caption with '{word.strip()}' {when_true:.0%} of the time when it is true, "
          f"and {when_false:.0%} of the time when it is false")

### ★ Q2.1 (10 pts)

(a) Report CLIP's accuracy on each challenge. Which challenges are at or below chance (0.5)?

(b) The last column compares each true caption with its foil using only CLIP's text encoder. What do values close to 1 tell you about where the failure happens? Why might contrastive training on web captions not force CLIP to encode word order?

(c) CLIP scores a little above chance on spatial and negation probes. Use the two lines printed above to argue whether it understands *above* and *no*.

**Your answer (Q2.1):**

*Write your answer here.*

## 2.2 A modular approach (12 pts autograded)

Modular systems, such as neural module networks (Andreas et al.) and ViperGPT, split the problem in two: perception, then reasoning.

1. **Find objects:** `find_objects` (given) returns a box for each blob of color.
2. **Perceive:** `perceive` classifies each object's color and shape. You write it, by running CLIP zero-shot on each crop with your Part 1 functions.
3. **Execute:** `execute` runs the caption's program on the object list. You write it, and it is autograded.

An object is `{"color": ..., "shape": ..., "box": (y0, x0, y1, x1)}`. A program is a nested dict. An object *matches* a spec when it agrees on the spec's `color` and `shape` keys (ignore any other keys, such as `n`):

| `op` | fields | true when |
|---|---|---|
| `exists` | `color` and/or `shape` (both optional) | some object matches |
| `count` | `color` and/or `shape` (both optional), and `n` | exactly `n` objects match |
| `relation` | `rel` (`"above"` or `"below"`), `a`, `b` | some object matching `a` is above (or below) some object matching `b`. Compare box centers; a strictly smaller y is higher. |
| `not` | `arg` | `arg` is false |
| `and` | `args` | every program in `args` is true |

In [ ]:
# GRADED: execute
def execute(program, objects):
    """Evaluate a program (a nested dict) on a list of objects. Returns True or False."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
from scipy import ndimage


def find_objects(img, min_area=30):
    """Boxes (y0, x0, y1, x1) of the connected non-white blobs in an image."""
    mask = np.asarray(img).astype(int).sum(-1) < 700
    labels, _ = ndimage.label(mask)
    boxes = [(s[0].start, s[1].start, s[0].stop, s[1].stop) for s in ndimage.find_objects(labels)]
    return [b for b in boxes if (b[2] - b[0]) * (b[3] - b[1]) >= min_area]


COLOR_W = build_zero_shot_weights(list(COLORS), ["a {} shape"], encode_text_b32)
SHAPE_W = build_zero_shot_weights(SHAPES, ["a {}"], encode_text_b32)


def perceive(img, pad=8):
    """Find the objects in `img` and classify each one's color and shape with CLIP zero-shot.
    Not autograded (its accuracy shows up in the table below).

    Returns a list of {"color", "shape", "box"} dicts, where "box" is the unpadded box from
    find_objects. Use encode_images on `crops`, then zero_shot_logits against COLOR_W and SHAPE_W.
    """
    boxes = find_objects(img)
    if not boxes:
        return []
    crops = [img.crop((max(x0 - pad, 0), max(y0 - pad, 0), min(x1 + pad, 224), min(y1 + pad, 224)))
             for y0, x0, y1, x1 in boxes]
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def modular_ok(probe, objects):
    return execute(probe["prog_true"], objects) and not execute(probe["prog_foil"], objects)

perceived = [perceive(p["image"]) for p in PROBES]
modular = np.array([modular_ok(p, objs) for p, objs in zip(PROBES, perceived)])
oracle = np.array([modular_ok(p, p["objects"]) for p in PROBES])
display(pd.DataFrame({"CLIP": [clip_ok[probe_kind == k].mean() for k in KINDS],
                      "modular": [modular[probe_kind == k].mean() for k in KINDS],
                      "modular, true objects": [oracle[probe_kind == k].mean() for k in KINDS]},
                     index=KINDS).round(3))

same = lambda a, b: sorted((o["color"], o["shape"]) for o in a) == sorted((o["color"], o["shape"]) for o in b)
exact = np.array([same(objs, p["objects"]) for p, objs in zip(PROBES, perceived)])
print(f"scenes perceived exactly right: {exact.mean():.1%}  (counting scenes: {exact[probe_kind == 'counting'].mean():.1%})")

### ★ Q2.2 (8 pts)

(a) Compare the modular system with CLIP on each challenge.

(b) With the true objects, execution is perfect. Where do the modular system's remaining errors come from? Use the perception rate printed above, and explain why counting suffers the most.

**Your answer (Q2.2):**

*Write your answer here.*

### ★ Q2.3 (10 pts)

The modular system only worked because we handed it a program for every caption.

(a) Write two captions about scenes like these that the program language above cannot express.

(b) Name one kind of concept or language for which an end-to-end model (CLIP or a VLM) is a better fit than a modular pipeline, and explain why.

(c) The lecture called this limitation the *code bottleneck*. Explain it in your own words, using one of your captions from (a).

**Your answer (Q2.3):**

*Write your answer here.*

---
# Part 3: Pragmatics with RSA (25 pts)

In a **reference game**, a speaker names a target among several objects, and a listener tries to pick it out. The Rational Speech Acts model (Frank & Goodman, 2012) starts from a literal score $\phi(u, o)$. For a hand-written lexicon, $\phi(u, o) = \log [\![u]\!](o)$, which is $-\infty$ when $u$ is false of $o$. For images, $\phi$ is a CLIP logit.

- $L_0(o \mid u) \propto \exp \phi(u, o)\; P(o)$
- $S_1(u \mid o) \propto \exp\big(\alpha\,(\log L_0(o \mid u) - C(u))\big)$
- $L_1(o \mid u) \propto S_1(u \mid o)\, P(o)$

## 3.1 Implement RSA in log space (10 pts, autograded)

Inputs and outputs are NumPy log-probabilities: `log_L0` and `log_L1` are `[U, O]`, and `log_S1` is `[O, U]`. `phi` may contain `-inf`, but every row and every column has a finite entry. `None` means a uniform prior or zero cost. Use a numerically stable log-softmax.

In [ ]:
# GRADED: rsa
import numpy as np


def _log_softmax(x, axis):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))


def rsa_L0(phi, log_prior=None):
    """phi: [U, O] literal scores (may contain -inf); log_prior: [O] or None. Returns log L0(o | u): [U, O]."""
    # YOUR CODE HERE
    raise NotImplementedError


def rsa_S1(log_L0, alpha=1.0, cost=None):
    """log_L0: [U, O]; alpha > 0; cost: [U] or None (zero cost). Returns log S1(u | o): [O, U]."""
    # YOUR CODE HERE
    raise NotImplementedError


def rsa_L1(log_S1, log_prior=None):
    """log_S1: [O, U]; log_prior: [O] or None. Returns log L1(o | u): [U, O]."""
    # YOUR CODE HERE
    raise NotImplementedError

### Frank & Goodman (2012)

The objects are a blue square, a blue circle and a green square. The utterances are *blue*, *green*, *square* and *circle*.

In [ ]:
FG_OBJECTS = ["blue square", "blue circle", "green square"]
FG_UTTS = ["blue", "green", "square", "circle"]
FG_LEXICON = np.array([[1, 1, 0],    # blue
                       [0, 0, 1],    # green
                       [1, 0, 1],    # square
                       [0, 1, 0]])   # circle
with np.errstate(divide="ignore"):
    fg_phi = np.log(FG_LEXICON)                              # -inf where the word is false

fg_L0 = rsa_L0(fg_phi)
fg_S1 = rsa_S1(fg_L0)
fg_L1 = rsa_L1(fg_S1)
show = lambda M, rows, cols: display(pd.DataFrame(np.exp(M), index=rows, columns=cols).round(3))
print("L0(o | u)"); show(fg_L0, FG_UTTS, FG_OBJECTS)
print("S1(u | o)"); show(fg_S1, FG_OBJECTS, FG_UTTS)
print("L1(o | u)"); show(fg_L1, FG_UTTS, FG_OBJECTS)
assert np.allclose(np.exp(fg_L1[0]), [0.6, 0.4, 0.0]), "L1('blue') should be [0.6, 0.4, 0] -- check your functions"
print("✓ L1('blue') = [0.6, 0.4, 0.0]")

### ★ Q3.1 (5 pts)

(a) Given *blue*, $L_0$ is indifferent between the two blue objects, but $L_1$ prefers the blue square. Explain why, using what the speaker *would have said* if the target were the blue circle.

(b) Which utterances does $L_1$ interpret exactly as $L_0$ does, and why?

**Your answer (Q3.1):**

*Write your answer here.*

## 3.2 A reference game with CLIP

Now $\phi(u, i)$ is the CLIP logit between the prompt for a single word $u$ and image $i$. The prompts are *a red shape*, ..., *a circle*, .... Each game shows 3 single-shape images, and image 0 is the target.

- **random** contexts: three different objects.
- **hard** contexts: one distractor shares the target's color and the other shares its shape, so no single word is true of the target alone.

The **context-blind** speaker $S_0$ says the word that best fits the target on its own, $\arg\max_u \phi(u, \text{target})$.

In [ ]:
def render_shape(color, shape, seed, size=224):
    rng = np.random.RandomState(seed)
    img = Image.new("RGB", (size, size), (255, 255, 255))
    r = rng.randint(45, 70)
    cy, cx = rng.randint(r + 10, size - r - 10, 2)[::-1]
    draw_shape(ImageDraw.Draw(img), color, shape, cy, cx, r)
    return img


OBJECTS = [(c, s) for c in COLORS for s in SHAPES]                       # 12 object types
ref_images, ref_object = [], []
for k, (c, s) in enumerate(OBJECTS):
    for j in range(5):
        ref_images.append(render_shape(c, s, seed=100 * k + j))
        ref_object.append(k)
ref_object = np.array(ref_object)
WORDS = list(COLORS) + SHAPES
ref_prompts = [f"a {w} shape" if w in COLORS else f"a {w}" for w in WORDS]
S_ref = LOGIT_SCALE * (F.normalize(encode_images(clip_b32, proc_b32, ref_images), dim=-1)
                       @ F.normalize(encode_texts(clip_b32, proc_b32, ref_prompts), dim=-1).T).numpy()   # [60, 7]


def sample_contexts(kind, n, seed):
    rng = np.random.RandomState(seed)
    out = []
    while len(out) < n:
        types = rng.choice(len(OBJECTS), 3, replace=False)
        (tc, ts), d1, d2 = [OBJECTS[t] for t in types]
        if kind == "hard" and not ((d1[0] == tc and d2[1] == ts) or (d2[0] == tc and d1[1] == ts)):
            continue
        out.append(np.array([rng.choice(np.where(ref_object == t)[0]) for t in types]))
    return out


CONTEXTS = {kind: sample_contexts(kind, 2000, seed) for seed, kind in enumerate(["random", "hard"])}
phi_of = lambda ctx: S_ref[ctx].T                                              # [U, C]
speakers = {"S0": lambda ctx: int(np.argmax(S_ref[ctx[0]])),
            "S1": lambda ctx: int(np.argmax(rsa_S1(rsa_L0(phi_of(ctx)))[0]))}
listeners = {"L0": lambda ctx, u: int(np.argmax(rsa_L0(phi_of(ctx))[u])),
             "L1": lambda ctx, u: int(np.argmax(rsa_L1(rsa_S1(rsa_L0(phi_of(ctx))))[u]))}
play = lambda spk, lis, contexts: float(np.mean([lis(ctx, spk(ctx)) == 0 for ctx in contexts]))

games = {}
for kind in ["random", "hard"]:
    for s, l in [("S0", "L0"), ("S1", "L0"), ("S1", "L1")]:
        games[(kind, f"{s} → {l}")] = play(speakers[s], listeners[l], CONTEXTS[kind])
display(pd.Series(games).unstack().round(3))

# one hard game where S0 fails and S1 succeeds with the literal listener
for ctx in CONTEXTS["hard"]:
    u0, u1 = speakers["S0"](ctx), speakers["S1"](ctx)
    if listeners["L0"](ctx, u0) != 0 and listeners["L0"](ctx, u1) == 0:
        fig, axes = plt.subplots(1, 3, figsize=(5.4, 2))
        for j, ax in enumerate(axes):
            ax.imshow(ref_images[ctx[j]]); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title("target" if j == 0 else "", fontsize=8)
        fig.suptitle(f"S0 says '{WORDS[u0]}', L0 picks image {listeners['L0'](ctx, u0)}.   "
                     f"S1 says '{WORDS[u1]}', L0 picks image 0.", fontsize=8)
        plt.tight_layout(); plt.show()
        break

### ★ Q3.2 (10 pts)

(a) Report the six numbers in the table.

(b) Using the example game, explain why $S_0$ fails in hard contexts.

(c) In hard contexts, why does switching the listener from $L_0$ to $L_1$ help $S_1$ so much? Relate your answer to Q3.1.

**Your answer (Q3.2):**

*Write your answer here.*

---
# Part 4 (extra credit, +10): Does a modern VLM fix Part 2?

LLaVA-style models (lecture) feed a CLIP-like vision encoder into an LLM through a projector, and are trained on visual instructions. The LLM reads the whole caption, word order included. Does that fix CLIP's failures from Part 2?

We use **SmolVLM-500M** (500M parameters) and never generate text. For each probe we ask

> Is this statement true for the image: "*caption*"? Answer yes or no.

and read the next-token logits. The score is the **yes/no margin** $\log P(\text{yes}) - \log P(\text{no})$, where each probability sums over several spellings (*Yes*, *␣yes*, …). A positive margin means the model says yes.

Part 4 is optional. Parts 1–3 do not depend on it.

## 4.1 The yes/no margin (4 pts, autograded)

In [ ]:
# GRADED: yes_no_margin
import torch


def yes_no_margin(logits, yes_ids, no_ids):
    """logits: [..., V] next-token logits; yes_ids, no_ids: lists of token ids.

    Returns a float tensor of shape logits.shape[:-1]:
    log(sum of p over yes_ids) - log(sum of p over no_ids), with p = softmax(logits) over the full vocabulary.
    Must stay finite for very large logits.
    """
    # YOUR CODE HERE
    raise NotImplementedError

## 4.2 The VLM on the Part 2 probes

The next cell downloads SmolVLM-500M (about 1 GB, about 2 GB of RAM) and scores 25 probes per challenge, each with its true caption and its foil. That is 200 forward passes: about 10 minutes on a free Colab CPU, and well under a minute on a GPU runtime. **If you skip Part 4, set `RUN_PART4 = False`** so that *Run all* skips the download.

In [ ]:
RUN_PART4 = True   # set to False if you skip Part 4

if RUN_PART4:
    from transformers import AutoModelForImageTextToText, AutoProcessor

    VLM_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"
    vlm_proc = AutoProcessor.from_pretrained(VLM_NAME, do_image_splitting=False)   # one tile per image: ~100 tokens
    vlm = AutoModelForImageTextToText.from_pretrained(
        VLM_NAME, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
    first_id = lambda w: vlm_proc.tokenizer.encode(w, add_special_tokens=False)[0]
    YES_IDS = sorted({first_id(w) for w in ["Yes", " Yes", "yes", " yes"]})
    NO_IDS = sorted({first_id(w) for w in ["No", " No", "no", " no"]})
    TEMPLATE = 'Is this statement true for the image: "{}"? Answer yes or no.'


    @torch.no_grad()
    def vlm_margin(image, statement, template=TEMPLATE):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": template.format(statement)}]}]
        x = vlm_proc(text=vlm_proc.apply_chat_template(msgs, add_generation_prompt=True), images=[image],
                     return_tensors="pt").to(device)
        return float(yes_no_margin(vlm(**x).logits[0, -1].float(), YES_IDS, NO_IDS))


    SUBSET = np.concatenate([np.where(probe_kind == k)[0][:25] for k in KINDS])
    t0 = time.time()
    m_true = np.array([vlm_margin(PROBES[i]["image"], PROBES[i]["true"]) for i in SUBSET])
    m_foil = np.array([vlm_margin(PROBES[i]["image"], PROBES[i]["foil"]) for i in SUBSET])
    print(f"{2 * len(SUBSET)} forward passes in {time.time() - t0:.0f} s")

    sub = probe_kind[SUBSET]
    per_kind = lambda ok: [ok[sub == k].mean() for k in KINDS]
    display(pd.DataFrame({"CLIP": per_kind(clip_ok[SUBSET]),
                          "modular": per_kind(modular[SUBSET]),
                          "VLM pairwise": per_kind(m_true > m_foil),
                          "VLM yes to true": per_kind(m_true > 0),
                          "VLM yes to foil": per_kind(m_foil > 0),
                          "VLM both answers right": per_kind((m_true > 0) & (m_foil < 0))}, index=KINDS).round(2))

*VLM pairwise* asks whether the true caption gets the higher margin, the same test we gave CLIP. *Both answers right* asks whether the model says yes to the true caption **and** no to the foil.

## 4.3 Does scale help?

We ran the same template on all 100 probes per challenge with larger VLMs on a GPU. The numbers are given below, so you don't need a GPU. Your SmolVLM-500M table uses only 25 probes per challenge, so it can differ from the SmolVLM-500M row here by about ±0.1.

In [ ]:
SCALE = {  # model: (pairwise, both answers right), challenges in the order of KINDS
    "SmolVLM-256M": ([0.56, 0.59, 0.73, 0.82], [0.15, 0.09, 0.18, 0.17]),
    "SmolVLM-500M": ([0.83, 0.87, 0.79, 0.97], [0.25, 0.07, 0.27, 0.33]),
    "SmolVLM2-2.2B": ([1.00, 0.84, 0.62, 1.00], [0.99, 0.05, 0.40, 0.99]),
    "Qwen2.5-VL-3B": ([1.00, 1.00, 0.81, 1.00], [0.98, 0.21, 0.52, 1.00]),
}
for j, label in enumerate(["pairwise", "both answers right"]):
    print(label)
    display(pd.DataFrame({m: v[j] for m, v in SCALE.items()}, index=KINDS).T)
print("Qwen2.5-VL-3B says yes to 79% of the false spatial captions.")

### ★ Q4 (6 pts)

(a) Using your table and the scale table, which challenges does a VLM fix compared with CLIP, and how does that depend on model size?

(b) Qwen2.5-VL-3B ranks every spatial pair correctly but gets both answers right on only 21% of them. What does this mean for using a VLM as a yes/no judge, for example as an automatic evaluator? How would you use it more reliably?

(c) Counting is still the weakest challenge for pairwise ranking at 3B (0.81). Is that more likely a vision-side or a language-side failure? Describe one experiment with these probes that would tell the two apart.

**Your answer (Q4):**

*Write your answer here.*

---
## Before you submit

1. *Runtime → Restart and run all.* Parts 1–3 must run. Part 4 is optional; if you skip it, set `RUN_PART4 = False` in 4.2.
2. Export `HW6.pdf` and check that the plots rendered.
3. Upload `HW6.ipynb` and `HW6.pdf` to Gradescope **A6**, or the `submission.zip` built by `python3 make_submission.py` (locally).